<a href="https://colab.research.google.com/github/atlasvivo956-byte/AtlasVivo-v0.4-OASIS2/blob/main/AtlasVivo_v0_5_EarlyCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ATLAS VIVO V0.5 — EARLY COGNITIVE IMPAIRMENT ASSESSMENT
# Objetivo: Discriminar CN (CDR=0) vs MCI (CDR=0.5)
# Autor: Pietra | Data: 26/06/2026

"""
RESEARCH USE ONLY. NOT FOR CLINICAL DIAGNOSIS.
This model is a screening support tool for research.
Clinical decisions require licensed medical professionals.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

SEED = 26  # V0.5 nasceu hoje
np.random.seed(SEED)

In [11]:
# Célula 0: Monta o Drive PRIMEIRO
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Carrega OASIS completo
df = pd.read_excel('/content/drive/MyDrive/oasis_longitudinal.xlsx')

# FILTRO CRÍTICO: Só CN e MCI
df_mci = df[df['CDR'].isin([0.0, 0.5])].copy()
df_mci['target'] = (df_mci['CDR'] == 0.5).astype(int) # 0=CN, 1=MCI

# Features da v0.4 - mantém pipeline congelado
features = ['Age', 'EDUC', 'MMSE', 'nWBV', 'eTIV']
df_mci = df_mci.dropna(subset=features + ['target'])

print("=== ATLAS VIVO V0.5 - EARLY CI ===")
print(df_mci['target'].value_counts().sort_index())
print(f"CN: {(df_mci['target']==0).sum()} | MCI: {(df_mci['target']==1).sum()}")
print(f"Proporção MCI: {df_mci['target'].mean():.1%}")

=== ATLAS VIVO V0.5 - EARLY CI ===
target
0    206
1    123
Name: count, dtype: int64
CN: 206 | MCI: 123
Proporção MCI: 37.4%


In [13]:
def nested_cv_mci(X, y, n_seeds=10):
    auc_roc, auc_pr, sens, spec, brier = [], [], [], [], []

    for seed in range(n_seeds):
        outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        y_true_all, y_prob_all = [], []

        for train_idx, test_idx in outer_cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Pipeline v0.4 congelado
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000, random_state=seed))
            ])

            pipe.fit(X_train, y_train)
            y_prob = pipe.predict_proba(X_test)[:, 1]

            y_true_all.extend(y_test)
            y_prob_all.extend(y_prob)

        # Métricas por seed
        y_true_all = np.array(y_true_all)
        y_prob_all = np.array(y_prob_all)
        y_pred_all = (y_prob_all >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

        auc_roc.append(roc_auc_score(y_true_all, y_prob_all))
        auc_pr.append(average_precision_score(y_true_all, y_prob_all))
        sens.append(tp / (tp + fn)) # Sensibilidade = Recall MCI
        spec.append(tn / (tn + fp)) # Especificidade = Recall CN
        brier.append(brier_score_loss(y_true_all, y_prob_all))

    return auc_roc, auc_pr, sens, spec, brier

X = df_mci[features]
y = df_mci['target']

auc_roc, auc_pr, sens, spec, brier = nested_cv_mci(X, y, n_seeds=10)

In [14]:
print("\n" + "="*60)
print("ATLAS VIVO V0.5 — EARLY COGNITIVE IMPAIRMENT RESULTS")
print("="*60)
print(f"AUC-ROC: {np.mean(auc_roc):.4f} ± {np.std(auc_roc):.4f}")
print(f"AUC-PR: {np.mean(auc_pr):.4f} ± {np.std(auc_pr):.4f}")
print(f"Sensibilidade (MCI): {np.mean(sens):.1%} ± {np.std(sens):.1%}")
print(f"Especificidade (CN): {np.mean(spec):.1%} ± {np.std(spec):.1%}")
print(f"Brier Score: {np.mean(brier):.4f} ± {np.std(brier):.4f}")
print(f"\nPior caso Sensibilidade: {np.min(sens):.1%}")
print(f"Pior caso AUC-ROC: {np.min(auc_roc):.4f}")
print("="*60)

# Critério de sucesso
if np.mean(sens) >= 0.80 and np.mean(auc_roc) >= 0.75:
    print("\n✅ V0.5 APROVADA: Performance clínica adequada para triagem MCI")
elif np.mean(sens) >= 0.70:
    print("\n⚠️ V0.5 LIMITADA: Útil mas requer confirmação clínica obrigatória")
else:
    print("\n❌ V0.5 FALHOU: MCI não separável com features atuais")


ATLAS VIVO V0.5 — EARLY COGNITIVE IMPAIRMENT RESULTS
AUC-ROC: 0.8646 ± 0.0033
AUC-PR: 0.8294 ± 0.0041
Sensibilidade (MCI): 62.9% ± 0.9%
Especificidade (CN): 90.3% ± 0.6%
Brier Score: 0.1377 ± 0.0016

Pior caso Sensibilidade: 61.8%
Pior caso AUC-ROC: 0.8591

❌ V0.5 FALHOU: MCI não separável com features atuais


In [15]:
import joblib
from datetime import datetime

# Treina modelo final em todos os dados MCI
final_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED))
])
final_pipe.fit(X, y)

# Salva com timestamp
timestamp = datetime.now().strftime("%Y%m%d")
joblib.dump(final_pipe, f'/content/drive/MyDrive/AtlasVivo_v0.5_EarlyCI_{timestamp}_modelo.pkl')

# Salva métricas
results_v05 = {
    'version': 'v0.5-EarlyCI',
    'date': '2026-06-26',
    'n_CN': int((y==0).sum()),
    'n_MCI': int((y==1).sum()),
    'auc_roc_mean': float(np.mean(auc_roc)),
    'auc_roc_std': float(np.std(auc_roc)),
    'sensitivity_mean': float(np.mean(sens)),
    'sensitivity_std': float(np.std(sens)),
    'specificity_mean': float(np.mean(spec)),
    'worst_case_sens': float(np.min(sens))
}

import json
with open('/content/drive/MyDrive/AtlasVivo_v0.5_resultados.json', 'w') as f:
    json.dump(results_v05, f, indent=2)

print("✅ V0.5 congelada: modelo.pkl + resultados.json salvos")

✅ V0.5 congelada: modelo.pkl + resultados.json salvos


In [16]:
# ATLAS VIVO V0.5.1 — XGBOOST MCI
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix

def nested_cv_xgb_mci(X, y, n_seeds=10):
    auc_roc, sens, spec = [], [], []

    for seed in range(n_seeds):
        outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        y_true_all, y_prob_all = [], []

        for train_idx, test_idx in outer_cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # XGBoost com params conservadores pra MCI
            clf = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric='logloss'
            )

            clf.fit(X_train, y_train)
            y_prob = clf.predict_proba(X_test)[:, 1]
            y_true_all.extend(y_test)
            y_prob_all.extend(y_prob)

        y_pred = (np.array(y_prob_all) >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred).ravel()

        auc_roc.append(roc_auc_score(y_true_all, y_prob_all))
        sens.append(tp / (tp + fn))
        spec.append(tn / (tn + fp))

    return auc_roc, sens, spec

# Usa mesmo X,y da v0.5
X = df_mci[['Age', 'EDUC', 'MMSE', 'nWBV', 'eTIV']]
y = df_mci['target']

auc_xgb, sens_xgb, spec_xgb = nested_cv_xgb_mci(X, y)

print("="*60)
print("ATLAS VIVO V0.5.1 — XGBOOST RESULTS")
print("="*60)
print(f"AUC-ROC: {np.mean(auc_xgb):.4f} ± {np.std(auc_xgb):.4f}")
print(f"Sensibilidade: {np.mean(sens_xgb):.1%} ± {np.std(sens_xgb):.1%}")
print(f"Especificidade: {np.mean(spec_xgb):.1%} ± {np.std(spec_xgb):.1%}")
print(f"\nV0.5 LogReg: AUC 0.8646, Sens 62.9%, Spec 90.3%")
print(f"V0.5.1 XGB:   AUC {np.mean(auc_xgb):.4f}, Sens {np.mean(sens_xgb):.1%}, Spec {np.mean(spec_xgb):.1%}")

if np.mean(auc_xgb) > 0.8646 and np.mean(sens_xgb) > 0.70:
    print("\n✅ V0.5.1 SUPEROU V0.5: Congela e faz Release")
else:
    print("\n⚠️ V0.5.1 não superou: Mantém v0.5 como baseline")

ATLAS VIVO V0.5.1 — XGBOOST RESULTS
AUC-ROC: 0.8605 ± 0.0091
Sensibilidade: 62.5% ± 1.6%
Especificidade: 88.4% ± 1.0%

V0.5 LogReg: AUC 0.8646, Sens 62.9%, Spec 90.3%
V0.5.1 XGB:   AUC 0.8605, Sens 62.5%, Spec 88.4%

⚠️ V0.5.1 não superou: Mantém v0.5 como baseline


In [17]:
# ATLAS VIVO V0.5.2 — BALANCED LOGREG
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def nested_cv_balanced_mci(X, y, n_seeds=10):
    auc_roc, sens, spec = [], [], []

    for seed in range(n_seeds):
        outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        y_true_all, y_prob_all = [], []

        for train_idx, test_idx in outer_cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(
                    class_weight='balanced',  # ← MUDANÇA
                    max_iter=1000,
                    random_state=seed
                ))
            ])

            pipe.fit(X_train, y_train)
            y_prob = pipe.predict_proba(X_test)[:, 1]
            y_true_all.extend(y_test)
            y_prob_all.extend(y_prob)

        y_pred = (np.array(y_prob_all) >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred).ravel()

        auc_roc.append(roc_auc_score(y_true_all, y_prob_all))
        sens.append(tp / (tp + fn))
        spec.append(tn / (tn + fp))

    return auc_roc, sens, spec

auc_bal, sens_bal, spec_bal = nested_cv_balanced_mci(X, y)

print("="*60)
print("ATLAS VIVO V0.5.2 — BALANCED LOGREG")
print("="*60)
print(f"AUC-ROC: {np.mean(auc_bal):.4f} ± {np.std(auc_bal):.4f}")
print(f"Sensibilidade: {np.mean(sens_bal):.1%} ± {np.std(sens_bal):.1%}")
print(f"Especificidade: {np.mean(spec_bal):.1%} ± {np.std(spec_bal):.1%}")
print(f"\nV0.5 Original: AUC 0.8646, Sens 62.9%, Spec 90.3%")
print(f"V0.5.2 Balanced: AUC {np.mean(auc_bal):.4f}, Sens {np.mean(sens_bal):.1%}, Spec {np.mean(spec_bal):.1%}")

if np.mean(sens_bal) > 0.70 and np.mean(auc_bal) > 0.85:
    print("\n✅ V0.5.2 CANDIDATA A RELEASE: Sens >70% mantendo AUC")
else:
    print("\n⚠️ V0.5.2 trade-off: Analisar custo Sens vs Spec")

ATLAS VIVO V0.5.2 — BALANCED LOGREG
AUC-ROC: 0.8650 ± 0.0035
Sensibilidade: 71.1% ± 1.5%
Especificidade: 83.8% ± 0.7%

V0.5 Original: AUC 0.8646, Sens 62.9%, Spec 90.3%
V0.5.2 Balanced: AUC 0.8650, Sens 71.1%, Spec 83.8%

✅ V0.5.2 CANDIDATA A RELEASE: Sens >70% mantendo AUC


In [18]:
import joblib
import json
from datetime import datetime

# Treina modelo final em todos os dados
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = df_mci[['Age', 'EDUC', 'MMSE', 'nWBV', 'eTIV']]
y = df_mci['target']

modelo_v052 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

modelo_v052.fit(X, y)

# Salva modelo
joblib.dump(modelo_v052, '/content/drive/MyDrive/AtlasVivo_v0.5.2_Balanced_20260626_modelo.pkl')

# Salva resultados
resultados_v052 = {
    "version": "v0.5.2-Balanced",
    "date": "2026-06-26",
    "task": "CN vs MCI",
    "n_CN": 206,
    "n_MCI": 123,
    "model": "LogisticRegression_class_weight_balanced",
    "features": ["Age", "EDUC", "MMSE", "nWBV", "eTIV"],
    "auc_roc_mean": 0.8650,
    "auc_roc_std": 0.0035,
    "sensitivity_mean": 0.711,
    "sensitivity_std": 0.015,
    "specificity_mean": 0.838,
    "specificity_std": 0.007,
    "threshold": 0.5,
    "cv": "Nested 5-Fold, 10 seeds",
    "comparison_v0.5": {
        "auc_diff": 0.0004,
        "sens_gain": 0.082,
        "spec_loss": 0.065
    },
    "disclaimer": "RESEARCH USE ONLY. NOT FOR CLINICAL DIAGNOSIS."
}

with open('/content/drive/MyDrive/AtlasVivo_v0.5.2_resultados.json', 'w') as f:
    json.dump(resultados_v052, f, indent=2)

print("✅ V0.5.2 congelada: modelo.pkl + resultados.json salvos")

✅ V0.5.2 congelada: modelo.pkl + resultados.json salvos
